In [3]:
import pandas as pd
df = pd.read_csv("/Users/annu/Desktop/Financial Misinformation Detection with LLM Hallucination Auditing + Market Impact Analysis/headlines_layer1.csv")

def prompt_variants(headline):
    return [
        f"""You are a financial fact-checking assistant. Classify this headline as FACTUAL or MISLEADING.
Headline: "{headline}"
Respond in this exact format:
CLASSIFICATION: <FACTUAL or MISLEADING>
CONFIDENCE: <HIGH or MEDIUM or LOW>
REASONING: <one sentence>""",

        f"""As a financial analyst, evaluate whether this headline accurately represents a real market event.
Headline: "{headline}"
Respond in this exact format:
CLASSIFICATION: <FACTUAL or MISLEADING>
CONFIDENCE: <HIGH or MEDIUM or LOW>
REASONING: <one sentence>""",

        f"""You are an expert in Indian and global financial markets. Determine if the following headline is misleading or factual.
Headline: "{headline}"
Respond in this exact format:
CLASSIFICATION: <FACTUAL or MISLEADING>
CONFIDENCE: <HIGH or MEDIUM or LOW>
REASONING: <one sentence>""",

        f"""Review this financial news headline for accuracy. Does it distort facts or manipulate market sentiment?
Headline: "{headline}"
Respond in this exact format:
CLASSIFICATION: <FACTUAL or MISLEADING>
CONFIDENCE: <HIGH or MEDIUM or LOW>
REASONING: <one sentence>""",

        f"""You are a SEBI-trained market surveillance expert. Assess whether this headline could mislead investors.
Headline: "{headline}"
Respond in this exact format:
CLASSIFICATION: <FACTUAL or MISLEADING>
CONFIDENCE: <HIGH or MEDIUM or LOW>
REASONING: <one sentence>"""
    ]

In [3]:
from groq import Groq
import pandas as pd
import time
from dotenv import load_dotenv
import os
from utils import parse_response 
from utils import generate_content
load_dotenv() #load .env files in this workspace
client = Groq(api_key=os.getenv("GROQ_API_KEY")) #so that api is hidden in github

def consistency_score(headline):
    prompts = prompt_variants(headline)
    response = [] #temporary list
    for p in prompts:
        raw_response = generate_content(p)
        parsed_response = parse_response(raw_response)
        response.append(parsed_response)
        time.sleep(2)
    classification = [r['classification']for r in response] #collecting only classification section in a list format
    majority = max(set(classification), key = classification.count)
    score = classification.count(majority)/ 5
    return score

df = pd.read_csv("/Users/annu/Desktop/Financial Misinformation Detection with LLM Hallucination Auditing + Market Impact Analysis/data/headlines_verified.csv")

score_list = []
for i, headline in enumerate(df['headline']):
    print(f"Processing headline {i+1}/87...")
    score = consistency_score(headline)
    score_list.append(score)





Processing headline 1/87...
Processing headline 2/87...
Processing headline 3/87...
Processing headline 4/87...
Processing headline 5/87...
Processing headline 6/87...
Processing headline 7/87...
Processing headline 8/87...
Processing headline 9/87...
Processing headline 10/87...
Processing headline 11/87...
Processing headline 12/87...
Processing headline 13/87...
Processing headline 14/87...
Processing headline 15/87...
Processing headline 16/87...
Processing headline 17/87...
Processing headline 18/87...
Processing headline 19/87...
Processing headline 20/87...
Processing headline 21/87...
Processing headline 22/87...
Processing headline 23/87...
Processing headline 24/87...
Processing headline 25/87...
Processing headline 26/87...
Processing headline 27/87...
Processing headline 28/87...
Processing headline 29/87...
Processing headline 30/87...
Processing headline 31/87...
Processing headline 32/87...
Processing headline 33/87...
Processing headline 34/87...
Processing headline 35/

In [8]:
from utils import build_prompt
from utils import parse_response
from dotenv import load_dotenv
import pandas as pd

load_dotenv()
client = Groq(api_key=os.getenv("GROQ_API_KEY"))

def generate_content_2(prompt): 
    chat = client.chat.completions.create(
        model="llama-3.3-70b-versatile", #different model of groq
        messages=[{"role": "user", "content": prompt}],
        temperature=0 )
    
    return chat.choices[0].message.content

response_2 = [] #respose classification from mixtral model
for index, row in df.iterrows(): #row wise iteration
    single_headline = row['headline']
    prompt = build_prompt(single_headline)
    row_response = generate_content_2(prompt)
    parsed = parse_response(row_response)
    response_2.append(parsed)
    time.sleep(1)

print(response_2)

df_layer1 = pd.read_csv("/Users/annu/Desktop/Financial Misinformation Detection with LLM Hallucination Auditing + Market Impact Analysis/headlines_layer1.csv")

df_mixtral = pd.DataFrame(response_2) #df of mixtral model response

df_mixtral = df_mixtral.rename( columns={
    'classification': 'mixtral_classification',
    'confidence':'mixtral_confidence',
    'reasoning' : 'mixtral_reasoning'
})

df_final = pd.concat([df_layer1,df_mixtral], axis=1)
df_final['consistency_score'] = score_list
df_final['agreement_score'] = (df_final['classification'] == df_final['mixtral_classification']).astype(int)
df_final.to_csv('headlines_layer2.csv', index=False)





[{'classification': 'FACTUAL', 'confidence': 'HIGH', 'reasoning': 'The headline accurately reflects a real event where Hindenburg Research released a report accusing Adani Group of stock manipulation and accounting fraud, using language that closely matches the content of the report.'}, {'classification': 'FACTUAL', 'confidence': 'HIGH', 'reasoning': "The headline accurately reports on a real event, specifically Adani Group's response to the Hindenburg report, and provides precise details about the rebuttal without using sensational language or omitting critical context."}, {'classification': 'FACTUAL', 'confidence': 'HIGH', 'reasoning': "The headline accurately reports a significant and verifiable financial event, specifically the Adani Group's substantial market value loss over a three-day period, without using sensational or misleading language."}, {'classification': 'FACTUAL', 'confidence': 'HIGH', 'reasoning': "The headline accurately reports a real event, specifically Adani Enter

In [4]:
df_layer1 = pd.read_csv('/Users/annu/Desktop/Financial Misinformation Detection with LLM Hallucination Auditing + Market Impact Analysis/headlines_layer1.csv')
print(df_layer1.columns.tolist())  # verify it only has layer1 columns

['date', 'headline', 'source', 'related_stock', 'category', 'news_type', 'classification', 'confidence', 'reasoning', 'classification.1', 'confidence.1', 'reasoning.1']


In [9]:
def calibration_flag(confidence, consistency_score):
    if confidence == 'HIGH' and consistency_score < 0.8:
        return 1
    else:
        return 0
    
import pandas as pd
df = pd.read_csv("/Users/annu/Desktop/Financial Misinformation Detection with LLM Hallucination Auditing + Market Impact Analysis/headlines_layer2.csv")

df['calibration_flag'] = df.apply(
    lambda row: calibration_flag(row['confidence'], row['consistency_score']), axis=1
)

df.to_csv('headlines_layer2.csv', index=False)



In [10]:
def hallucination_risk_score(consistency, agreement, calibration):
    score = 1 - (
        0.4 * consistency +
        0.4 * agreement +
        0.2 * (1 - calibration)
    )
    return round(score, 4)
import pandas as pd
df = pd.read_csv("/Users/annu/Desktop/Financial Misinformation Detection with LLM Hallucination Auditing + Market Impact Analysis/headlines_layer2.csv")

df['hallucination_score'] = df.apply(
    lambda row: hallucination_risk_score(row['consistency_score'],row['agreement_score'],row['calibration_flag']), axis = 1
)

df.to_csv('headlines_layer2.csv', index=False)